# Running on Modal

## What you'll learn

- Deploy a command operation as a Modal endpoint
- Run the same tool locally and remotely
- Verify remote execution from its output artifacts
- Observe concurrent endpoint calls

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Compute Routing](01-compute-routing.ipynb), and a Modal account with credentials.
**Estimated time:** 15 minutes
**GPU required:** No; the example runs on CPU containers.

This optional tutorial makes billable cloud calls.

:::{note}
Set up Modal authentication and deploy `wait_tool` before running the remote
cells. The steps below link to the deployment guide. The local tutorial path
does not require a Modal account.
:::

In [ ]:
from __future__ import annotations

from artisan.operations.examples import DataGenerator, WaitTool
from artisan.orchestration import PipelineManager, StepDisposition
from artisan.schemas import ComputeProvider, ModalComputeConfig
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_data, inspect_pipeline, inspect_step

In [ ]:
env = tutorial_setup("modal_execution", clean=True)
DELTA_ROOT = env.delta_root

## Use a command operation

`WaitTool` wraps a Bash command. It counts once per second and writes a CSV
marker containing the duration, source filename, and container task ID. Those
markers let us check where each call ran.

A command operation declares a `ToolSpec` and returns its command from
`execute_command()`. The local provider runs that command as a subprocess;
the Modal provider sends it to the deployed endpoint. Plain Python
`execute_function()` operations run locally. A function operation can opt into
a command wrapper with `execute_as_tool=True`; this tutorial uses an explicit
command operation.

We use the bundled operation from `src/artisan/operations/examples/wait_tool.py`.
[Writing Creator Operations](../../how-to-guides/writing-creator-operations.md)
shows how to author a command operation.

## Deploy the endpoint

Complete the setup in [Deploy Tool Endpoints](../../how-to-guides/deploying-tool-endpoints.md),
then deploy the bundled operation:

```bash
pixi run --locked -e dev artisan modal deploy wait_tool
```

This creates the `artisan-tool-wait_tool` Modal app. Deployment reads the
operation’s class configuration, including its image, hardware, secrets, scaling,
and URI policy. Redeploy after changing those values or the operation code.
The default URI policy permits inline file transfer and denies remote URI reads
and writes.

## Authenticate the client

Create Modal proxy-auth tokens as described in the
[deployment guide](../../how-to-guides/deploying-tool-endpoints.md). Put them in
the source checkout’s gitignored `.env` file or the process environment:

```dotenv
MODAL_PROXY_TOKEN_ID=wk-...
MODAL_PROXY_TOKEN_SECRET=ws-...
```

Artisan checks the environment first, then the nearest `.env` file above the
working directory. Restart or update your notebook environment if it already
contains different token values.

## Choose deployment hardware

This example uses CPUs. For a GPU tool, declare class-level resources such as
`compute_resources = ComputeResources(gpu="A100")` before deploying it.
Changing remote hardware requires redeployment: a per-step `compute_resources`
override cannot resize an existing endpoint. See
[Deploy Tool Endpoints](../../how-to-guides/deploying-tool-endpoints.md) for the
complete configuration.

In [ ]:
config = ComputeProvider(active="modal", modal=ModalComputeConfig())

print(f"Active provider: {config.active}")
print(f"Available:       {config.available()}")
print(f"Worker image:    {config.modal.image}")
print(f"Poll interval:   {config.modal.poll_interval}s")

## Running a step on the endpoint

`compute_provider="modal"` is the only change — the operation, params, and
output wiring stay identical. `skip_cache=True` ensures each example
executes, including when you rerun the cell:


In [ ]:
pipeline = PipelineManager.create(
    name="modal_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

gen = pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 2, "seed": 42},
)

# local first — same op, no Modal required
local_wait = pipeline.run(
    operation=WaitTool,
    skip_cache=True,
    name="wait_local",
    inputs={"dataset": gen.output("datasets")},
    params={"seconds": 1},
)

# then on the deployed endpoint — one argument changed
modal_wait = pipeline.run(
    operation=WaitTool,
    skip_cache=True,
    name="wait_modal",
    inputs={"dataset": gen.output("datasets")},
    params={"seconds": 1},
    compute_provider="modal",
)

summary = pipeline.finalize()
assert summary["overall_success"], summary
assert local_wait.disposition is StepDisposition.EXECUTED
assert modal_wait.disposition is StepDisposition.EXECUTED
for completed_step in (gen, local_wait, modal_wait):
    artifacts = inspect_step(
        DELTA_ROOT,
        completed_step.step_number,
        pipeline_run_id=pipeline.config.pipeline_run_id,
    )
    assert artifacts.height == 2, (completed_step.step_name, artifacts)
print(f"Pipeline complete: success={summary['overall_success']}")
inspect_pipeline(DELTA_ROOT, pipeline_run_id=pipeline.config.pipeline_run_id)

## Read the result

Both calls produced two marker artifacts. The local and Modal steps have
disposition `EXECUTED`, so the example performed real work rather than reusing
a cached result. For the remote call, the client sent the inputs and parameters,
waited for completion, and downloaded the outputs. The worker then recorded
the artifacts and their lineage.

Tool logs arrive with the result. Use the Modal dashboard or app logs to watch
progress while a call is running. For transport limits, URI policies, and
cancellation details, see [Deploy Tool Endpoints](../../how-to-guides/deploying-tool-endpoints.md).

## Watch endpoint calls overlap

The next step groups eight artifacts into one execution unit. Default
per-artifact dispatch still makes eight endpoint calls, which can overlap.
Modal decides how many containers are available; a container handles one job
at a time. A warm pool can reduce startup delay, but requires deployment-time
configuration.

Unit size also determines the execution-level cache and failure boundary.
Use smaller units when independent reuse or failure handling matters more than
shared setup. [Configure Execution](../../how-to-guides/configuring-execution.md)
explains the concurrency settings.

For remote GPUs, configure the endpoint’s `compute_resources.gpu` and redeploy.
A local `runner_resources.gpus` request does not reserve remote hardware and
reduces default local worker concurrency to one unless `max_workers` is explicit.

Eight sequential ten-second waits take at least 80 seconds. Concurrent calls can
overlap, but startup time and capacity determine the duration you observe.

In [ ]:
scale = PipelineManager.create(
    name="modal_scale_out",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = scale.output

scale_gen = scale.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 8, "seed": 42},
)

# One unit can issue up to eight concurrent endpoint calls.
step = scale.run(
    operation=WaitTool,
    skip_cache=True,
    name="wait",
    inputs={"dataset": output("generate", "datasets")},
    params={"seconds": 10},
    compute_provider="modal",
    batch_strategy={"artifacts_per_unit": 8},
)

scale_summary = scale.finalize()
assert scale_summary["overall_success"], scale_summary
assert step.disposition is StepDisposition.EXECUTED
for completed_step in (scale_gen, step):
    artifacts = inspect_step(
        env.delta_root,
        completed_step.step_number,
        pipeline_run_id=scale.config.pipeline_run_id,
    )
    assert artifacts.height == 8, (completed_step.step_name, artifacts)
print("All 8 remote waits produced an output artifact.")

### Watching it run

While the cell above executes (the first run also pays for container
cold starts):

- **Modal dashboard** — open the `artisan-tool-wait_tool` app: the
  container count may climb as the autoscaler fans out, and each container's log shows its ticks
  arriving one per second.
- **CLI** — in a terminal:

  ```bash
  modal app logs artisan-tool-wait_tool
  ```

  Overlapping calls show interleaved ticks from different container task
  IDs. The autoscaler controls how many containers actually run.

### Proof in the artifacts

Each container wrote its own task id into its marker file, so the
committed artifacts identify which containers handled the eight calls.
Count their host values below; the exact number depends on available
capacity and container reuse.


In [ ]:
markers = inspect_data(
    env.delta_root,
    step_number=step.step_number,
    pipeline_run_id=scale.config.pipeline_run_id,
)
assert markers.height == 8
assert markers["seconds"].to_list() == [10] * 8
assert all(markers["host"].to_list())
hosts = set(markers["host"].to_list())
print(f"distinct containers observed: {len(hosts)}")
print(sorted(hosts))

## Call the endpoint from another client

The deployed endpoint also accepts HTTP clients outside Artisan. Its schema
describes input roles and parameter validation; clients submit work, poll for
completion, then download the output archive. Use the request example in
[Deploy Tool Endpoints](../../how-to-guides/deploying-tool-endpoints.md) when
connecting another application.

## Diagnose a failed remote call

Check the persisted step result and failure report first. Confirm that the
endpoint is deployed, the proxy-auth tokens are valid, and the deployed code
matches the local operation. Run with `compute_provider="local"` to isolate
command behavior from endpoint configuration.

For endpoint-specific failures and expired results, follow
[Deploy Tool Endpoints](../../how-to-guides/deploying-tool-endpoints.md).

## Summary

You deployed a command operation, ran it locally and on Modal, and checked the
accepted output artifacts. The second example sent eight per-artifact calls
from one execution unit and inspected the container IDs in their markers.
The pipeline’s output wiring stays the same when you select remote compute.

## Next steps

- [Compute Routing](01-compute-routing.ipynb) — Step runners vs compute providers
- [Configure Execution](../../how-to-guides/configuring-execution.md) — Execution configuration recipes
- [Execution Flow](../../concepts/execution-flow.md) — How the framework dispatches and tracks work

- [Per-Artifact Batch Execute](../04-batching/02-batch-execute.ipynb) — Compare endpoint concurrency limits
- [Delivering Outputs to Object Storage](05-modal-r2-outputs.ipynb) — Deliver outputs to a bucket